# 🚀 Agent Runtime 배포 실습 (`agent_runtime.ipynb`)

외부 API(환율 조회 `get_exchange_rate`)를 호출하는 로컬 ADK 에이전트를 정의하고, **Vertex AI Agent Runtime (`client.runtimes.create`)**에 서버리스 컨테이너로 배포한 뒤 실시간 스트리밍 질의(`async_stream_query`)를 수행하는 노트북입니다.

---

## 💡 실습 흐름 요약

1. **에이전트 및 도구(Tool) 정의 (`Develop an agent`)**:
   - Python 함수(`get_exchange_rate`)를 도구로 등록한 ADK `Agent`를 생성하고, 이를 `AdkApp(agent=agent)`으로 래핑합니다.
2. **Agent Runtime 원격 배포 (`Deploy an agent to Agent Runtime`)**:
   - `client.runtimes.create()`를 호출하면 로컬의 `AdkApp` 객체가 직렬화(`cloudpickle`)되어 Staging GCS 버킷에 업로드되고, 지정된 `requirements`와 환경변수(`env_vars`), 서비스 계정(`service_account`)을 갖춘 관리형 런타임 컨테이너가 프로비저닝됩니다.
3. **원격 에이전트 스트리밍 호출 (`Test the Agent on Agent Runtime`)**:
   - 배포된 `remote_agent.async_stream_query()`를 통해 도구 호출(`function_call` ➔ `function_response`)과 최종 답변이 스트리밍되는 과정을 확인합니다.

In [ ]:
# %pip 매직 커맨드를 사용하여 현재 Jupyter 커널의 Python 환경에 패키지를 설치합니다. (루트 uv sync 환경을 커널로 선택한 경우 생략 가능)
%pip install --upgrade "google-cloud-aiplatform[agent_engines,adk]" cloudpickle pydantic python-dotenv requests

# Develop an agent

In [ ]:
import os
from dotenv import load_dotenv
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents import Agent

# 루트 .env 파일 로드
load_dotenv(override=True)
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")

def get_exchange_rate(
    currency_from: str = "USD",
    currency_to: str = "EUR",
    currency_date: str = "latest",
):
    """Retrieves the exchange rate between two currencies on a specified date."""
    import requests

    response = requests.get(
        f"https://api.frankfurter.app/{currency_date}",
        params={"from": currency_from, "to": currency_to},
    )
    return response.json()

# Agent 및 App 정의
agent = Agent(
    model=GEMINI_MODEL,
    name="currency_exchange_agent",
    tools=[get_exchange_rate],
)

app = AdkApp(agent=agent)

# Deploy an agent to Agent Runtime

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT") or input("Enter PROJECT_ID: ").strip()
LOCATION = os.getenv("GCP_RESOURCES_LOCATION", "us-central1")
STAGING_BUCKET = os.getenv("STAGING_BUCKET_URI") or input("Enter STAGING_BUCKET_URI (e.g. gs://...): ").strip()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
SERVICE_ACCOUNT = os.getenv("SERVICE_ACCOUNT", f"sa-agent-platform@{PROJECT_ID}.iam.gserviceaccount.com")

print(f"Project: {PROJECT_ID}, Location: {LOCATION}, Bucket: {STAGING_BUCKET}, SA: {SERVICE_ACCOUNT}")

In [ ]:
import agentplatform

client = agentplatform.Client(
    project=PROJECT_ID,
    location=LOCATION,  
)

In [ ]:
remote_agent = client.runtimes.create(
    agent=app,
    config={
        "display_name": "currency_exchange_agent",
        "service_account": SERVICE_ACCOUNT,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]",
            "cloudpickle",
            "pydantic",
            "requests",
        ],
        "staging_bucket": STAGING_BUCKET,
        "env_vars": {
            "GOOGLE_CLOUD_LOCATION": "global",
            "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
            "GEMINI_MODEL": GEMINI_MODEL,
        },
    },
)

# Test the Agent on Agent Runtime

In [ ]:
async for event in remote_agent.async_stream_query(
    user_id="USER_ID",
    message="What is the exchange rate from US dollars to SEK today?",
):
    print(event)

In [ ]:
# 실습 완료 후 배포된 Reasoning Engine 삭제 (필요 시 주석 해제 후 실행)
# remote_agent.delete(force=True)